# Marketing A/B Test: Ad vs. PSA — Conversion Analysis

A retrospective analysis of a real marketing experiment (Kaggle "Marketing A/B Testing" dataset,
588,101 users), evaluating whether showing users an ad — instead of a public service announcement
(PSA) placeholder — increased purchase conversion.

This project deliberately contrasts with a companion project (MFI / Giné & Karlan microfinance
replication): randomization here happens at the **individual user level**, not a cluster/center
level, so no cluster-robust standard errors or cluster-level permutation are required. The goal
of this project is to practice the full A/B testing workflow end-to-end on a clean, unclustered
design, and to be explicit about which steps of a standard A/B testing framework can and cannot
be executed properly on *historical* (already-collected) data versus a live, prospective test.

Analysis is organized around a 7-step A/B testing framework:
1. Problem Statement
2. Define Success Metric / Hypothesis Testing
3. Design the Experiment
4. Run the Experiment
5. Validity Checks
6. Interpret Results
7. Launch Decision

Each step below states explicitly whether it was executed on real data, or whether it could only
be addressed conceptually / with illustrative assumptions, given this is historical data.

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [2]:
df = pd.read_csv('../data/marketing_AB.csv')
df.head(10)

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14
5,5,1137664,ad,False,734,Saturday,10
6,6,1116205,ad,False,264,Wednesday,13
7,7,1496843,ad,False,17,Sunday,18
8,8,1448851,ad,False,21,Tuesday,19
9,9,1446284,ad,False,142,Monday,14


## Step 1 — Problem Statement

Since this is historical data, the goal is reconstructed rather than newly set: **does showing a
user an ad (vs. a neutral PSA) increase the rate at which they convert?**

## Step 2 — Define Success Metric & Hypothesis Testing

**Success metric — `converted`:**
- *Measurable* — binary field, already logged per user.
- *Attributable* — plausible, since exposure was (as far as we can tell) randomly assigned; this
  is checked properly in Step 5's Sample Ratio Mismatch test below.
- *Sensitive* — conversion is a low-probability event (~2%), so a large sample is needed; we have
  588k rows.
- *Timely* — recorded within the dataset's own window; no lag issue.

**Hypotheses:**
- H0: conversion rate(ad) = conversion rate(psa)
- H1: conversion rate(ad) != conversion rate(psa)
- alpha = 0.05, power = 80% (conventional choices)

**MDE — calculated retrospectively, not set prospectively.** Because n is already fixed at
588,101 (unequal split, ~96% ad / 4% psa), MDE is solved *from* the existing sample size rather
than used to solve *for* a sample size (the reverse of how Step 3's formula is normally used in a
prospective design).

In [3]:
# Step 2: Achieved MDE, given the actual (unequal) sample sizes already collected
ad = df.loc[df['test group'] == 'ad', 'converted']
psa = df.loc[df['test group'] == 'psa', 'converted']

p1, n1 = ad.mean(), len(ad)
p2, n2 = psa.mean(), len(psa)

z_alpha, z_beta = 1.96, 0.84  # alpha=0.05 two-sided, power=80%
se_mde = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
mde = (z_alpha + z_beta) * se_mde

print(f'Achieved MDE given actual sample sizes: {mde:.4%}')

Achieved MDE given actual sample sizes: 0.2488%


**Achieved MDE ≈ 0.25 percentage points.** With this sample size, a true conversion-rate gap
as small as ~0.25pp could have been reliably detected. Note the psa group (23,524 rows, the
minority side of the unequal split) is the binding constraint on precision — it contributes far
more to the variance sum than the much larger ad group, illustrating that an unbalanced split
makes the *smaller* group the statistical bottleneck.

## Step 3 — Design the Experiment

- **Randomization unit:** individual user (no cluster/group in between subject and assignment).
- **Target population:** all users the site's test logic exposed to this experiment (no evidence
  of further filtering available in this file).
- **Sample size:** already fixed by history at n=588,101 (Step 2 above used this formula in
  reverse — solving for MDE given n, rather than n given a chosen MDE).
- **Duration:** not addressed — the dataset carries no date field, so the experiment's actual
  calendar window cannot be recovered.

## Step 4 — Run the Experiment

- **Instrumentation / data pipeline:** not applicable retrospectively — collection already
  happened, and no pipeline documentation is available for this file.
- **δ (practical significance threshold) — illustrative only.** This dataset has no revenue or
  cost fields, so δ cannot be derived from the data; it can only be estimated from assumed
  external costs, stated explicitly as an assumption:

In [4]:
# Step 4: Illustrative breakeven delta (NOT derived from this dataset — assumed costs)
cost_per_impression = 0.02   # ASSUMPTION, not from data
profit_per_conversion = 15   # ASSUMPTION, not from data

delta_breakeven = cost_per_impression / profit_per_conversion
print(f'Illustrative breakeven delta: {delta_breakeven:.4%}')

Illustrative breakeven delta: 0.1333%


**δ ≈ 0.13 percentage points, under these illustrative assumptions only.** This number should
not be treated as a real business threshold — it exists purely to demonstrate the *logic* of
comparing a statistically detectable effect (MDE) against a financially meaningful one (δ), which
are two conceptually distinct thresholds even though both answer "how small an effect matters."

## Step 5 — Validity Checks

Each bias/check from the framework, addressed individually and honestly where a check could not
be run:

- **Sample Ratio Mismatch** — checked below, real data.
- **Instrumentation Effect** (does the ad itself technically break the page, independent of any
  persuasion effect?) — not checked; this dataset has no guardrail metric (page load time, error
  rate) available.
- **External Factors** (holidays, competitor activity) — not checked; no date field available.
- **Selection Bias** — normally checked via a live A/A test, which is unavailable retrospectively.
  A partial substitute is attempted below via a covariate balance check on `total ads`.
- **Novelty Effect** (new vs. returning visitors) — not checked; no visitor-history field
  available in this file.

### Sample Ratio Mismatch (SRM)

In [5]:
observed = df['test group'].value_counts()
print(observed)

# Intended ratio is an ASSUMPTION based on the dataset's known public description (~96/4),
# since the original experiment's design document is not available.
intended_ratio = {'ad': 0.96, 'psa': 0.04}
total = observed.sum()
expected = pd.Series({k: v * total for k, v in intended_ratio.items()})

chi2, p_srm = chisquare(f_obs=observed[expected.index], f_exp=expected)
print(f'chi2 = {chi2:.4e}, p = {p_srm:.4f}')

test group
ad     564577
psa     23524
Name: count, dtype: int64
chi2 = 7.0850e-08, p = 0.9998


**SRM result: chi2 ≈ 7.08e-08, p ≈ 0.9998.** The observed split matches the assumed intended
96/4 ratio almost exactly — this is the reassuring direction for an SRM test (unlike most
hypothesis tests, a *large* p-value here is the good outcome). No evidence of broken or biased
group assignment.

## Step 6 — Interpret Results

### Parametric test (two-proportion z-test)

Relies on the Central Limit Theorem: each user's conversion is a Bernoulli outcome, and with
large samples the difference between two group conversion *rates* is well-approximated by a
normal distribution, letting the p-value come from a known formula rather than simulation.

In [6]:
count = np.array([ad.sum(), psa.sum()])
nobs = np.array([len(ad), len(psa)])

z_stat, p_value = proportions_ztest(count, nobs)
print(f'conversion rate (ad):  {ad.mean():.4%}')
print(f'conversion rate (psa): {psa.mean():.4%}')
print(f'z-statistic: {z_stat:.4f}')
print(f'p-value: {p_value:.4e}')

conversion rate (ad):  2.5547%
conversion rate (psa): 1.7854%
z-statistic: 7.3701
p-value: 1.7053e-13


### Non-parametric cross-check (permutation test)

Makes no assumption about the shape of the sampling distribution — instead builds a reference
distribution directly from the data by repeatedly shuffling the `converted` labels across all
rows (individual-level shuffle, since randomization here is individual-level, not cluster-level).

**Correction applied:** the original notebook version of this cell allocated `stats =
np.empty(n_perm)` but never wrote `new_ad` into it inside the loop, so the printed p-value was
computed from uninitialized memory, not from the actual permutations. Fixed below by assigning
`stats[i] = new_ad` on every iteration.

In [7]:
rng = np.random.default_rng(seed=42)
observed_diff = ad.mean() - psa.mean()

y = df['converted'].to_numpy()
n_ad = len(ad)
n_perm = 10000
stats = np.empty(n_perm)

for i in range(n_perm):
    shuffled = rng.permutation(y)
    stats[i] = shuffled[:n_ad].mean() - shuffled[n_ad:].mean()   # FIX: was not assigned before

p_perm = (np.sum(np.abs(stats) >= np.abs(observed_diff)) + 1) / (n_perm + 1)
print(f'observed diff: {observed_diff:.6f}')
print(f'permutation p-value: {p_perm:.4e}')

observed diff: 0.007692
permutation p-value: 9.9990e-05


Result: **p ≈ 9.999×10⁻⁵** — this lands on the floor resolvable by
10,000 permutations (1/10,001), consistent with the z-test's far smaller p-value (~1.7×10⁻¹³);
none of the 10,000 random reshuffles produced a gap as large as the real one. Parametric and
non-parametric methods agree, which is the expected cross-validation outcome once the sample is
this large and the effect this clear.

In [8]:
se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
ci_low = observed_diff - 1.96 * se
ci_high = observed_diff + 1.96 * se
relative_lift = observed_diff / p2

print(f'Absolute lift: {observed_diff:.4%}')
print(f'95% CI: ({ci_low:.4%}, {ci_high:.4%})')
print(f'Relative lift: {relative_lift:.2%}')

Absolute lift: 0.7692%
95% CI: (0.5951%, 0.9434%)
Relative lift: 43.09%


**95% CI ≈ (0.59pp, 0.94pp); relative lift ≈ 43%.** Even the pessimistic (lower) bound of the
CI clears the illustrative δ (0.13pp) from Step 4 by a wide margin. Absolute (pp) and relative (%)
framings describe the same number from two different reference points — both are reported
together deliberately, since citing only the relative figure without the absolute baseline is a
common way results get oversold.

### Covariate balance check — `total ads` (partial substitute for Selection Bias / A/A test)

`total ads` is the only other behavioral field in this dataset. It was tested for two different
purposes:
1. As a potential per-user *cost driver* for Step 7's launch decision.
2. As indirect evidence of balanced randomization (Step 5), in place of an unavailable A/A test.

In [9]:
summary = df.groupby('test group')['total ads'].agg(['mean', 'median', 'std', 'count'])
print(summary)

ad_ads = df.loc[df['test group'] == 'ad', 'total ads']
psa_ads = df.loc[df['test group'] == 'psa', 'total ads']

stat, p_mw = mannwhitneyu(ad_ads, psa_ads, alternative='two-sided')
print(f'\nMann-Whitney U = {stat:.4e}, p = {p_mw:.4e}')

                mean    median       std   count
test group                                      
ad         24.823365 13.000000 43.750456  564577
psa        24.761138 12.000000 42.860720   23524

Mann-Whitney U = 6.8083e+09, p = 4.6909e-11


**Result: means/medians are nearly identical (24.82 vs 24.76; median 13 vs 12), but
Mann-Whitney returns p ≈ 4.69e-11 — statistically significant.**

This is a **statistical-significance-vs-practical-significance split**, the same underlying issue
as MDE (Step 2) vs. δ (Step 4), now showing up in a covariate-balance check instead of the primary
outcome: with 564,577 vs. 23,524 rows, the test has enough power to flag a one-ad difference in
the median as "significant," even though that gap is negligible in any practical sense.

**Conclusions drawn from this, kept separate:**
- **For Step 7 (cost driver):** `total ads` does not respond to treatment in any way that could
  function as a real per-user cost estimate — likely measures general site-wide ad exposure
  rather than an outcome of this specific test. This trade-off check is a confirmed dead end for
  that purpose.
- **For Step 5 (balance substitute):** the near-identical means/medians are still reasonable
  partial evidence of balanced randomization on this one dimension, once the practical-vs-
  statistical distinction above is accounted for — a "statistically imbalanced, practically
  balanced" verdict, not an unqualified "balanced" one.

## Step 7 — Launch Decision

- **Statistical result:** the ad effect is essentially certain to be real (z-test p ≈ 1.7e-13; CI
  entirely positive and well clear of the illustrative δ).
- **Metric trade-offs:** the only other trackable metric (`total ads`) could not serve as a cost
  signal (see above) — no evidence found of a metric moving in a negative direction alongside the
  conversion gain, but this is a limited check given the dataset's available fields.
- **Cost of launching:** "launching" has effectively already happened (96% of users were already
  shown ads); the real open question is whether reaching the remaining 4% (psa) group at scale
  would produce diminishing returns — not answerable from this single-snapshot dataset.

**Honest bottom line:** statistically, the effect is real and large relative to what this sample
size could detect. Whether to act on it is entirely conditional on the true cost/profit numbers
behind δ, which are not available in this dataset. The correct framing for any stakeholder-facing
summary is *"conditional on these cost assumptions, the lift clears breakeven,"* not an
unconditional "launch."

## Summary of what was actually verified vs. assumed

| Item | Status |
|---|---|
| SRM check | Verified, real data |
| Z-test (parametric) | Verified, real data |
| Permutation test (non-parametric) | Verified, real data (p ≈ 9.999e-05) |
| 95% CI, relative lift | Computed from verified z-test inputs |
| Covariate balance (`total ads`) | Verified, real data |
| MDE (Step 2) | Computed from real sample sizes |
| δ (Step 4) | **Illustrative only** — no real cost/revenue data available |
| Duration, Instrumentation, External Factors, Novelty Effect | **Not checked** — fields unavailable in this dataset |
